# Week 10 — Module 21: Bayesian Optimisation

Surrogate validation, prediction accuracy, acquisition comparisons and function-specific refinement.

## Load observations

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Locate the previous week's cumulative datasets.
search_roots = [Path.cwd(), *Path.cwd().parents]
relative_locations = [
    Path("9. Module 20/updated_data_week_9"),
    Path("updated_data_week_9"),
    Path("10. Module 21/Data"),
    Path("Data"),
]
expected_before = {1:18, 2:18, 3:23, 4:38, 5:28, 6:28, 7:38, 8:48}
expected_after = {i: count + 1 for i, count in expected_before.items()}

def suitable(folder):
    if not all((folder / f"function_{i}.csv").is_file() for i in range(1, 9)):
        return False
    counts = {i: len(pd.read_csv(folder / f"function_{i}.csv")) for i in range(1, 9)}
    return counts == expected_before or counts == expected_after

data_folder = next(
    (base / relative for base in search_roots for relative in relative_locations
     if suitable(base / relative)),
    None,
)
if data_folder is None:
    raise FileNotFoundError("Could not find the eight Week 9 cumulative function CSVs.")

data = {i: pd.read_csv(data_folder / f"function_{i}.csv") for i in range(1, 9)}
display(pd.DataFrame([
    {"Function": f"F{i}", "Observations": len(data[i]), "Dimensions": data[i].shape[1] - 1}
    for i in range(1, 9)
]))

,Function,Observations,Dimensions
0,F1,19,2
1,F2,19,2
2,F3,24,3
3,F4,39,4
4,F5,29,4
5,F6,29,5
6,F7,39,6
7,F8,49,8


## Week 9 observations

In [2]:
week9_inputs = {
    1: [0.682178, 0.001903],
    2: [0.704305, 0.038527],
    3: [0.473523, 0.500295, 0.456235],
    4: [0.388825, 0.366595, 0.350533, 0.420635],
    5: [1.000000, 0.969998, 1.000000, 1.000000],
    6: [0.440497, 0.372759, 0.525613, 0.791828, 0.000000],
    7: [0.199451, 0.330731, 0.460263, 0.255214, 0.346569, 0.603374],
    8: [0.031565, 0.175185, 0.160043, 0.130670,
        0.742356, 0.487463, 0.194164, 0.622246]
}

week9_outputs = {
    1: 7.209519565393848e-171,
    2: 0.5572172325218441,
    3: -0.011197594399301977,
    4: 0.5837298256552903,
    5: 8115.40044651129,
    6: -0.2397692962261966,
    7: 2.8391956973154033,
    8: 9.9849814248974
}

## Week 9 results

In [3]:
week9_results = []
for i in range(1, 9):
    df = data[i]
    x_cols, y_col = list(df.columns[:-1]), df.columns[-1]
    point = np.array(week9_inputs[i], dtype=float)
    observed = df[x_cols].to_numpy(dtype=float)
    matches = np.flatnonzero(np.all(np.isclose(observed, point, atol=5e-7, rtol=0), axis=1))
    if len(matches):
        saved_output = float(df.iloc[matches[0]][y_col])
        if not np.isclose(saved_output, week9_outputs[i], rtol=1e-7, atol=1e-9):
            raise ValueError(f"F{i}: existing observation has a different output")
        previous_best = float(df.drop(index=df.index[matches[0]])[y_col].max())
    else:
        previous_best = float(df[y_col].max())
    week9_results.append({"Function": f"F{i}", "Previous best": previous_best,
                          "Week 9 output": week9_outputs[i],
                          "Difference": week9_outputs[i] - previous_best,
                          "Improved": week9_outputs[i] > previous_best})
display(pd.DataFrame(week9_results).round(6))

,Function,Previous best,Week 9 output,Difference,Improved
0,F1,0.000000,0.000000,-0.000000,False
1,F2,0.746962,0.557217,-0.189745,False
2,F3,-0.000863,-0.011198,-0.010335,False
3,F4,0.387262,0.583730,0.196468,True
4,F5,8662.482500,8115.400447,-547.082053,False
5,F6,-0.274388,-0.239769,0.034619,True
6,F7,2.929370,2.839196,-0.090174,False
7,F8,9.957032,9.984981,0.027950,True


## Update observations

In [4]:
for i in range(1, 9):
    df = data[i]
    x_cols, y_col = list(df.columns[:-1]), df.columns[-1]
    point = np.array(week9_inputs[i], dtype=float)
    X = df[x_cols].to_numpy(dtype=float)
    matches = np.flatnonzero(np.all(np.isclose(X, point, atol=5e-7, rtol=0), axis=1))
    if not len(matches):
        row = dict(zip(x_cols, point))
        row[y_col] = week9_outputs[i]
        data[i] = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
display(pd.DataFrame([
    {"Function": f"F{i}", "Observations": len(data[i]), "Dimensions": data[i].shape[1] - 1}
    for i in range(1, 9)
]))

,Function,Observations,Dimensions
0,F1,19,2
1,F2,19,2
2,F3,24,3
3,F4,39,4
4,F5,29,4
5,F6,29,5
6,F7,39,6
7,F8,49,8


## Realised prediction accuracy

In [5]:
# Predictions recorded for the exact Round 10 candidates submitted
week9_predictions = {
    1: {"GP": None, "NN": None},
    2: {"GP": None, "NN": 0.8101565208},
    3: {"GP": -0.0001656607, "NN": -0.0007295399},
    4: {"GP": 0.5639568598, "NN": 0.3110049422},
    5: {"GP": 8287.6230170929, "NN": 8688.6043182866},
    6: {"GP": -0.1958705160, "NN": -0.2297615776},
    7: {"GP": 2.9288912234, "NN": 2.9853999933},
    8: {"GP": 9.9762750495, "NN": 10.0096962780}
}


accuracy_rows = []
for i in range(1, 9):
    actual = week9_outputs[i]
    gp_pred, nn_pred = week9_predictions[i]["GP"], week9_predictions[i]["NN"]
    gp_error = abs(gp_pred - actual) if gp_pred is not None else np.nan
    nn_error = abs(nn_pred - actual) if nn_pred is not None else np.nan
    closer = ("GP" if gp_error < nn_error else "NN" if nn_error < gp_error else "Equal") if (gp_pred is not None and nn_pred is not None) else "N/A"
    accuracy_rows.append({"Function": f"F{i}", "Actual": actual, "GP prediction": gp_pred,
                          "GP absolute error": gp_error, "NN prediction": nn_pred,
                          "NN absolute error": nn_error, "Closer model": closer})
display(pd.DataFrame(accuracy_rows).round(6))

,Function,Actual,GP prediction,GP absolute error,NN prediction,NN absolute error,Closer model
0,F1,0.000000,NaN,NaN,NaN,NaN,N/A
1,F2,0.557217,NaN,NaN,0.810157,0.252939,N/A
2,F3,-0.011198,-0.000166,0.011032,-0.000730,0.010468,NN
3,F4,0.583730,0.563957,0.019773,0.311005,0.272725,GP
4,F5,8115.400447,8287.623017,172.222571,8688.604318,573.203872,GP
5,F6,-0.239769,-0.195871,0.043899,-0.229762,0.010008,NN
6,F7,2.839196,2.928891,0.089696,2.985400,0.146204,GP
7,F8,9.984981,9.976275,0.008706,10.009696,0.024715,GP


## Gaussian Process models

In [6]:
import numpy as np
import warnings
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.exceptions import ConvergenceWarning
gp_models = {}
for i in range(1, 9):
    df = data[i]
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    kernel = RBF(length_scale=1.0, length_scale_bounds=(1e-05, 100000.0))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=42)
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter('always', ConvergenceWarning)
        gp.fit(X, y)
    gp_models[i] = gp
    for warning in caught_warnings:
        if issubclass(warning.category, ConvergenceWarning):
            pass

display(pd.DataFrame([
    {"Function": f"F{i}", "Observations": len(data[i]),
     "RBF length scale": float(gp_models[i].kernel_.length_scale),
     "Log marginal likelihood": gp_models[i].log_marginal_likelihood(gp_models[i].kernel_.theta),
     "At lower bound": bool(np.isclose(gp_models[i].kernel_.length_scale, 1e-5, rtol=1e-3))}
    for i in range(1, 9)
]).round(6))

,Function,Observations,RBF length scale,Log marginal likelihood,At lower bound
0,F1,19,0.000010,-26.959832,True
1,F2,19,0.000010,-26.959832,True
2,F3,24,0.096756,-29.946737,False
3,F4,39,0.316203,-19.214108,False
4,F5,29,0.338323,-20.623561,False
5,F6,29,0.517560,-20.566977,False
6,F7,39,0.319519,-32.767946,False
7,F8,49,1.071095,-18.081197,False


## Neural-network LOOCV

In [7]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
nn_loocv_results = {}
for i in range(1, 9):
    df = data[i]
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    loo = LeaveOneOut()
    actual_values = []
    predicted_values = []
    for train_index, test_index in loo.split(X):
        X_train, X_test = (X[train_index], X[test_index])
        y_train, y_test = (y[train_index], y[test_index])
        nn = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(50, 50), activation='relu', solver='lbfgs', max_iter=5000, random_state=42))
        nn.fit(X_train, y_train)
        prediction = nn.predict(X_test)[0]
        actual_values.append(y_test[0])
        predicted_values.append(prediction)
    nn_mae = mean_absolute_error(actual_values, predicted_values)
    baseline_predictions = []
    for train_index, test_index in loo.split(X):
        y_train = y[train_index]
        baseline_predictions.append(np.mean(y_train))
    baseline_mae = mean_absolute_error(y, baseline_predictions)
    beats_baseline = nn_mae < baseline_mae
    nn_loocv_results[i] = {'nn_mae': nn_mae, 'baseline_mae': baseline_mae, 'beats_baseline': beats_baseline}

display(pd.DataFrame([
    {"Function": f"F{i}", "Observations": len(data[i]),
     "NN LOOCV MAE": nn_loocv_results[i]["nn_mae"],
     "Baseline MAE": nn_loocv_results[i]["baseline_mae"],
     "Beats baseline": nn_loocv_results[i]["beats_baseline"]}
    for i in range(1, 9)
]).round(6))

,Function,Observations,NN LOOCV MAE,Baseline MAE,Beats baseline
0,F1,19,0.033792,0.000946,False
1,F2,19,0.203415,0.238444,True
2,F3,24,0.076948,0.054294,False
3,F4,39,1.221272,7.632505,True
4,F5,29,220.544899,2556.291575,True
5,F6,29,0.161145,0.547373,True
6,F7,39,0.348938,0.777671,True
7,F8,49,0.520153,0.999338,True


## GP acquisition candidates

In [8]:
import numpy as np
from scipy.stats import norm
np.random.seed(42)
ucb_kappas = [0.5, 1.0, 1.5, 2.5]
ei_percentages = [0.0, 0.01, 0.05, 0.1]
gp_candidates = {}
for i in range(1, 9):
    df = data[i]
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    gp = gp_models[i]
    n_dims = X.shape[1]
    candidates = np.random.uniform(0, 1, size=(100000, n_dims))
    mu, sigma = gp.predict(candidates, return_std=True)
    current_best = np.max(y)
    gp_candidates[i] = {}
    for kappa in ucb_kappas:
        acquisition = mu + kappa * sigma
        idx = np.argmax(acquisition)
        point = candidates[idx]
        pred = mu[idx]
        std = sigma[idx]
        nearest_distance = np.min(np.linalg.norm(X - point, axis=1))
        label = f'UCB_{kappa}'
        gp_candidates[i][label] = {'point': point, 'predicted': pred, 'uncertainty': std, 'nearest_distance': nearest_distance}
    for pct in ei_percentages:
        improvement_target = current_best + abs(current_best) * pct
        improvement = mu - improvement_target
        with np.errstate(divide='ignore', invalid='ignore'):
            Z = np.divide(improvement, sigma, out=np.zeros_like(improvement), where=sigma > 0)
        ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0] = 0
        idx = np.argmax(ei)
        point = candidates[idx]
        pred = mu[idx]
        std = sigma[idx]
        nearest_distance = np.min(np.linalg.norm(X - point, axis=1))
        pct_label = int(pct * 100)
        label = f'EI_{pct_label}pct'
        gp_candidates[i][label] = {'point': point, 'predicted': pred, 'uncertainty': std, 'nearest_distance': nearest_distance}

gp_rows = []
for i in range(1, 9):
    for method, result in gp_candidates[i].items():
        gp_rows.append({"Function": f"F{i}", "Method": method,
                        "Candidate": np.array2string(result["point"], precision=6, separator=", ", max_line_width=200),
                        "GP prediction": result["predicted"], "Uncertainty": result["uncertainty"],
                        "Nearest distance": result["nearest_distance"]})
display(pd.DataFrame(gp_rows).round(6))

,Function,Method,Candidate,GP prediction,Uncertainty,Nearest distance
0,F1,UCB_0.5,"[0.37454 , 0.950714]",-0.000501,0.001508,0.195683
1,F1,UCB_1.0,"[0.37454 , 0.950714]",-0.000501,0.001508,0.195683
2,F1,UCB_1.5,"[0.37454 , 0.950714]",-0.000501,0.001508,0.195683
3,F1,UCB_2.5,"[0.37454 , 0.950714]",-0.000501,0.001508,0.195683
4,F1,EI_0pct,"[0.37454 , 0.950714]",-0.000501,0.001508,0.195683
...,...,...,...,...,...,...
59,F8,UCB_2.5,"[0.510895, 0.131916, 0.028423, 0.942942, 0.702...",9.401114,0.473939,0.906832
60,F8,EI_0pct,"[0.510895, 0.131916, 0.028423, 0.942942, 0.702...",9.401114,0.473939,0.906832
61,F8,EI_1pct,"[0.510895, 0.131916, 0.028423, 0.942942, 0.702...",9.401114,0.473939,0.906832
62,F8,EI_5pct,"[0.510895, 0.131916, 0.028423, 0.942942, 0.702...",9.401114,0.473939,0.906832


## Neural-network candidates

In [9]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
nn_candidates = {}
for i in range(1, 9):
    df = data[i]
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    n_dims = X.shape[1]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    models = []
    for seed in [1, 2, 3, 4, 5]:
        nn = MLPRegressor(hidden_layer_sizes=(50, 50), activation='relu', solver='lbfgs', max_iter=5000, random_state=seed)
        nn.fit(X_scaled, y)
        models.append(nn)
    candidates = np.random.uniform(0, 1, size=(100000, n_dims))
    candidates_scaled = scaler.transform(candidates)
    predictions = np.array([model.predict(candidates_scaled) for model in models])
    mean_pred = predictions.mean(axis=0)
    disagreement = predictions.std(axis=0)
    score = mean_pred - disagreement
    idx = np.argmax(score)
    point = candidates[idx]
    pred = mean_pred[idx]
    disagree = disagreement[idx]
    nearest_distance = np.min(np.linalg.norm(X - point, axis=1))
    validated = nn_loocv_results[i]['beats_baseline']
    nn_candidates[i] = {'point': point, 'predicted': pred, 'disagreement': disagree, 'nearest_distance': nearest_distance, 'validated': validated}

display(pd.DataFrame([
    {"Function": f"F{i}",
     "Candidate": np.array2string(nn_candidates[i]["point"], precision=6, separator=", ", max_line_width=200),
     "NN prediction": nn_candidates[i]["predicted"],
     "Disagreement": nn_candidates[i]["disagreement"],
     "Nearest distance": nn_candidates[i]["nearest_distance"],
     "LOOCV validated": nn_candidates[i]["validated"]}
    for i in range(1, 9)
]).round(6))

,Function,Candidate,NN prediction,Disagreement,Nearest distance,LOOCV validated
0,F1,"[0.014679, 0.660492]",0.015695,0.006535,0.265804,False
1,F2,"[0.998959, 0.845004]",0.995563,0.112410,0.138158,True
2,F3,"[0.46704 , 0.511353, 0.462257]",-0.005204,0.001166,0.014162,False
3,F4,"[0.396518, 0.359299, 0.344171, 0.472153]",0.111120,0.107599,0.052981,True
4,F5,"[0.961746, 0.962953, 0.996451, 0.931679]",6212.672864,99.199510,0.054547,True
5,F6,"[0.451376, 0.372011, 0.613511, 0.790532, 0.031...",-0.273307,0.012744,0.094106,True
6,F7,"[0.154221, 0.344873, 0.523185, 0.277056, 0.348...",2.898103,0.055071,0.059970,True
7,F8,"[0.042126, 0.035099, 0.292494, 0.21494 , 0.040...",10.433905,0.273262,0.677331,True


## GP–NN candidate comparison

In [10]:
import numpy as np
gp_nn_agreement = {}
for i in range(1, 9):
    nn_point = nn_candidates[i]['point']
    distances = []
    for method, details in gp_candidates[i].items():
        gp_point = details['point']
        distance = np.linalg.norm(nn_point - gp_point)
        distances.append((method, distance))
    distances.sort(key=lambda x: x[1])
    gp_nn_agreement[i] = distances
    for method, distance in distances:
        pass
    closest_method, closest_distance = distances[0]

agreement_rows = []
for i in range(1, 9):
    row = {"Function": f"F{i}"}
    for method, dist in gp_nn_agreement[i]:
        row[method] = dist
    agreement_rows.append(row)
display(pd.DataFrame(agreement_rows).round(6))

,Function,UCB_0.5,UCB_1.0,UCB_1.5,UCB_2.5,EI_0pct,EI_1pct,EI_5pct,EI_10pct
0,F1,0.462308,0.462308,0.462308,0.462308,0.462308,0.462308,0.462308,0.462308
1,F2,0.813902,0.813902,0.813902,0.813902,0.813902,0.813902,0.813902,0.813902
2,F3,0.061613,0.073589,0.120126,0.169131,0.051914,0.051914,0.051914,0.051914
3,F4,0.197230,0.243764,0.361906,0.657291,0.197230,0.197230,0.197230,0.197230
4,F5,0.000000,0.000000,0.000000,1.631172,1.631172,1.631172,1.631172,1.631172
5,F6,0.170772,0.308435,0.308435,1.207496,0.308435,0.308435,0.308435,0.308435
6,F7,0.246767,0.246767,0.307181,0.331133,0.246767,0.246767,0.307181,0.331133
7,F8,0.949228,1.002629,1.389577,1.389577,1.389577,1.389577,1.389577,1.389577


## Neighbourhood evidence

In [11]:
neighbour_rows = []
for i in range(1, 9):
    X = data[i].iloc[:, :-1].to_numpy(dtype=float)
    y = data[i].iloc[:, -1].to_numpy(dtype=float)
    candidates_to_check = {
        "NN": nn_candidates[i]["point"],
        "UCB_0.5": gp_candidates[i]["UCB_0.5"]["point"],
        "UCB_2.5": gp_candidates[i]["UCB_2.5"]["point"],
    }
    for method, point in candidates_to_check.items():
        distances = np.linalg.norm(X - point, axis=1)
        nearest_indices = np.argsort(distances)[:3]
        neighbour_rows.append({"Function": f"F{i}", "Method": method,
                               "Candidate": np.array2string(point, precision=6, separator=", ", max_line_width=200),
                               **{f"Neighbour {k} output": y[idx] for k, idx in enumerate(nearest_indices, 1)},
                               **{f"Neighbour {k} distance": distances[idx] for k, idx in enumerate(nearest_indices, 1)}})
display(pd.DataFrame(neighbour_rows).round(6))

,Function,Method,Candidate,Neighbour 1 output,Neighbour 2 output,Neighbour 3 output,Neighbour 1 distance,Neighbour 2 distance,Neighbour 3 distance
0,F1,NN,"[0.014679, 0.660492]",0.000000,0.000000,0.000000,0.265804,0.321491,0.336695
1,F1,UCB_0.5,"[0.37454 , 0.950714]",0.000000,0.000000,0.000000,0.195683,0.211968,0.321627
2,F1,UCB_2.5,"[0.37454 , 0.950714]",0.000000,0.000000,0.000000,0.195683,0.211968,0.321627
3,F2,NN,"[0.998959, 0.845004]",0.420586,0.293993,-0.082041,0.138158,0.203823,0.222931
4,F2,UCB_0.5,"[0.282588, 0.458677]",-0.065624,0.214965,-0.013858,0.177755,0.240630,0.251146
5,F2,UCB_2.5,"[0.282588, 0.458677]",-0.065624,0.214965,-0.013858,0.177755,0.240630,0.251146
6,F3,NN,"[0.46704 , 0.511353, 0.462257]",-0.011198,-0.000863,-0.010190,0.014162,0.016794,0.060001
7,F3,UCB_0.5,"[0.409006, 0.529329, 0.472508]",-0.010190,-0.000863,-0.011198,0.055414,0.066263,0.072597
8,F3,UCB_2.5,"[0.335021, 0.528501, 0.566575]",-0.010190,-0.020167,-0.016451,0.142024,0.156635,0.166057
9,F4,NN,"[0.396518, 0.359299, 0.344171, 0.472153]",0.583730,0.387262,0.186912,0.052981,0.054625,0.065667


## Multi-radius local refinement

In [12]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
local_functions = [3, 4, 6, 7, 8]
radii = [0.03, 0.05, 0.1]
local_refinement = {}
for i in local_functions:
    df = data[i]
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_output = y[best_idx]
    gp = gp_models[i]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    nn_models = []
    for seed in [1, 2, 3, 4, 5]:
        nn = MLPRegressor(hidden_layer_sizes=(50, 50), activation='relu', solver='lbfgs', max_iter=5000, random_state=seed)
        nn.fit(X_scaled, y)
        nn_models.append(nn)
    local_refinement[i] = {}
    for radius in radii:
        n_candidates = 100000
        n_dims = X.shape[1]
        perturbations = np.random.uniform(-radius, radius, size=(n_candidates, n_dims))
        candidates = best_point + perturbations
        candidates = np.clip(candidates, 0, 1)
        gp_mean, gp_std = gp.predict(candidates, return_std=True)
        candidates_scaled = scaler.transform(candidates)
        nn_predictions = np.array([model.predict(candidates_scaled) for model in nn_models])
        nn_mean = nn_predictions.mean(axis=0)
        nn_std = nn_predictions.std(axis=0)
        gp_scale = np.std(y)
        if gp_scale == 0:
            gp_scale = 1.0
        gp_norm = (gp_mean - np.mean(gp_mean)) / (np.std(gp_mean) + 1e-12)
        nn_norm = (nn_mean - np.mean(nn_mean)) / (np.std(nn_mean) + 1e-12)
        gp_uncertainty_norm = gp_std / gp_scale
        nn_disagreement_norm = nn_std / gp_scale
        score = gp_norm + nn_norm - 0.25 * gp_uncertainty_norm - 0.25 * nn_disagreement_norm
        idx = np.argmax(score)
        point = candidates[idx]
        distance_from_best = np.linalg.norm(point - best_point)
        nearest_existing = np.min(np.linalg.norm(X - point, axis=1))
        local_refinement[i][radius] = {'point': point, 'gp_predicted': gp_mean[idx], 'gp_uncertainty': gp_std[idx], 'nn_predicted': nn_mean[idx], 'nn_disagreement': nn_std[idx], 'distance_from_best': distance_from_best, 'nearest_existing': nearest_existing}

local_rows = []
for i, radius_results in local_refinement.items():
    for radius, result in radius_results.items():
        local_rows.append({"Function": f"F{i}", "Radius": radius,
                           "Candidate": np.array2string(result["point"], precision=6, separator=", ", max_line_width=200),
                           "GP prediction": result["gp_predicted"],
                           "GP uncertainty": result["gp_uncertainty"],
                           "NN prediction": result["nn_predicted"],
                           "NN disagreement": result["nn_disagreement"],
                           "Distance from best": result["distance_from_best"],
                           "Nearest distance": result["nearest_existing"]})
display(pd.DataFrame(local_rows).round(6))

,Function,Radius,Candidate,GP prediction,GP uncertainty,NN prediction,NN disagreement,Distance from best,Nearest distance
0,F3,0.03,"[0.440788, 0.482725, 0.435914]",0.011428,0.006525,-0.000707,0.008639,0.032341,0.032341
1,F3,0.05,"[0.440641, 0.482965, 0.435994]",0.011539,0.006384,-0.000729,0.008567,0.032293,0.032293
2,F3,0.10,"[0.438834, 0.485909, 0.437512]",0.012920,0.004589,-0.001088,0.007583,0.031661,0.031661
3,F4,0.03,"[0.3905 , 0.365368, 0.351642, 0.422492]",0.594489,0.004702,0.568474,0.028776,0.002998,0.002998
4,F4,0.05,"[0.391916, 0.362517, 0.34972 , 0.419983]",0.616477,0.018431,0.545166,0.037792,0.005222,0.005222
5,F4,0.10,"[0.387857, 0.365827, 0.350733, 0.415753]",0.604620,0.010547,0.540369,0.041938,0.005040,0.005040
6,F6,0.03,"[0.44407 , 0.359961, 0.516223, 0.761872, 0. ...",-0.215407,0.017533,-0.230151,0.013006,0.034089,0.034089
7,F6,0.05,"[0.444874, 0.347937, 0.53347 , 0.742972, 0. ...",-0.209104,0.026683,-0.230632,0.020226,0.055533,0.055533
8,F6,0.10,"[0.452666, 0.347442, 0.532488, 0.727016, 0. ...",-0.204221,0.033342,-0.232288,0.028529,0.070971,0.070971
9,F7,0.03,"[0.150914, 0.296887, 0.488599, 0.266801, 0.336...",2.978986,0.063110,2.956245,0.029617,0.046855,0.046855


## Function 5: novelty-constrained refinement

In [13]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
i = 5
df = data[i]
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
best_idx = np.argmax(y)
best_point = X[best_idx]
best_output = y[best_idx]
gp = gp_models[i]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
nn_models = []
for seed in [1, 2, 3, 4, 5]:
    nn = MLPRegressor(hidden_layer_sizes=(50, 50), activation='relu', solver='lbfgs', max_iter=5000, random_state=seed)
    nn.fit(X_scaled, y)
    nn_models.append(nn)
minimum_novelties = [0.03, 0.05, 0.1]
function5_refinement = {}
for min_novelty in minimum_novelties:
    n_candidates = 200000
    n_dims = X.shape[1]
    candidates = np.random.uniform(0.8, 1.0, size=(n_candidates, n_dims))
    boundary_mask = np.random.random(size=(n_candidates, n_dims)) < 0.5
    candidates[boundary_mask] = 1.0
    nearest_distances = np.min(np.linalg.norm(candidates[:, None, :] - X[None, :, :], axis=2), axis=1)
    valid_mask = nearest_distances >= min_novelty
    valid_candidates = candidates[valid_mask]
    valid_distances = nearest_distances[valid_mask]
    if len(valid_candidates) == 0:
        continue
    gp_mean, gp_std = gp.predict(valid_candidates, return_std=True)
    valid_scaled = scaler.transform(valid_candidates)
    nn_predictions = np.array([model.predict(valid_scaled) for model in nn_models])
    nn_mean = nn_predictions.mean(axis=0)
    nn_std = nn_predictions.std(axis=0)
    gp_norm = (gp_mean - np.mean(gp_mean)) / (np.std(gp_mean) + 1e-12)
    nn_norm = (nn_mean - np.mean(nn_mean)) / (np.std(nn_mean) + 1e-12)
    output_scale = np.std(y)
    if output_scale == 0:
        output_scale = 1.0
    gp_uncertainty_norm = gp_std / output_scale
    nn_disagreement_norm = nn_std / output_scale
    score = gp_norm + nn_norm - 0.25 * gp_uncertainty_norm - 0.25 * nn_disagreement_norm
    idx = np.argmax(score)
    point = valid_candidates[idx]
    distance_from_best = np.linalg.norm(point - best_point)
    function5_refinement[min_novelty] = {'point': point, 'gp_predicted': gp_mean[idx], 'gp_uncertainty': gp_std[idx], 'nn_predicted': nn_mean[idx], 'nn_disagreement': nn_std[idx], 'distance_from_best': distance_from_best, 'nearest_existing': valid_distances[idx]}

display(pd.DataFrame([
    {"Minimum novelty": novelty,
     "Candidate": np.array2string(result["point"], precision=6, separator=", ", max_line_width=200),
     "GP prediction": result["gp_predicted"], "GP uncertainty": result["gp_uncertainty"],
     "NN prediction": result["nn_predicted"], "NN disagreement": result["nn_disagreement"],
     "Distance from best": result["distance_from_best"],
     "Nearest distance": result["nearest_existing"]}
    for novelty, result in function5_refinement.items()
]).round(6))

,Minimum novelty,Candidate,GP prediction,GP uncertainty,NN prediction,NN disagreement,Distance from best,Nearest distance
0,0.03,"[1. , 1. , 1. , 0.969985]",8156.394720,48.806856,8124.168392,44.564870,0.030015,0.030015
1,0.05,"[1. , 1. , 1. , 0.949992]",7793.218387,83.398553,7783.653892,57.605125,0.050008,0.050008
2,0.10,"[1. , 1. , 1. , 0.894864]",6717.968992,219.056976,6904.108750,50.006541,0.105136,0.100003


## Function 1: space-filling exploration

In [14]:
import numpy as np
np.random.seed(42)
i = 1
df = data[i]
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
n_candidates = 200000
n_dims = X.shape[1]
candidates = np.random.uniform(0, 1, size=(n_candidates, n_dims))
nearest_distances = np.min(np.linalg.norm(candidates[:, None, :] - X[None, :, :], axis=2), axis=1)
idx = np.argmax(nearest_distances)
function1_candidate = candidates[idx]
nearest_distance = nearest_distances[idx]
distances_to_existing = np.linalg.norm(X - function1_candidate, axis=1)
nearest_indices = np.argsort(distances_to_existing)[:3]
for rank, obs_idx in enumerate(nearest_indices, start=1):
    pass

display(pd.DataFrame([
    {"Candidate": np.array2string(function1_candidate, precision=6, separator=", "),
     "Nearest-observation distance": nearest_distance,
     "Neighbour 1 distance": distances_to_existing[nearest_indices[0]],
     "Neighbour 2 distance": distances_to_existing[nearest_indices[1]],
     "Neighbour 3 distance": distances_to_existing[nearest_indices[2]]}
]).round(6))

,Candidate,Nearest-observation distance,Neighbour 1 distance,Neighbour 2 distance,Neighbour 3 distance
0,"[0.002495, 0.694432]",0.301746,0.301746,0.302489,0.324234


## Function 2: NN-supported refinement

In [15]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
i = 2
df = data[i]
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
best_idx = np.argmax(y)
best_point = X[best_idx]
best_output = y[best_idx]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
nn_models = []
for seed in [1, 2, 3, 4, 5]:
    nn = MLPRegressor(hidden_layer_sizes=(50, 50), activation='relu', solver='lbfgs', max_iter=5000, random_state=seed)
    nn.fit(X_scaled, y)
    nn_models.append(nn)
radii = [0.03, 0.05, 0.1]
function2_refinement = {}
for radius in radii:
    n_candidates = 200000
    n_dims = X.shape[1]
    directions = np.random.normal(size=(n_candidates, n_dims))
    direction_norms = np.linalg.norm(directions, axis=1, keepdims=True)
    directions = directions / (direction_norms + 1e-12)
    candidates = best_point + radius * directions
    candidates = np.clip(candidates, 0, 1)
    candidates_scaled = scaler.transform(candidates)
    predictions = np.array([model.predict(candidates_scaled) for model in nn_models])
    mean_pred = predictions.mean(axis=0)
    disagreement = predictions.std(axis=0)
    score = mean_pred - disagreement
    idx = np.argmax(score)
    point = candidates[idx]
    distance_from_best = np.linalg.norm(point - best_point)
    nearest_existing = np.min(np.linalg.norm(X - point, axis=1))
    function2_refinement[radius] = {'point': point, 'nn_predicted': mean_pred[idx], 'nn_disagreement': disagreement[idx], 'distance_from_best': distance_from_best, 'nearest_existing': nearest_existing}

display(pd.DataFrame([
    {"Radius": radius, "Candidate": np.array2string(result["point"], precision=6, separator=", "),
     "NN prediction": result["nn_predicted"], "NN disagreement": result["nn_disagreement"],
     "Distance from best": result["distance_from_best"],
     "Nearest distance": result["nearest_existing"]}
    for radius, result in function2_refinement.items()
]).round(6))

,Radius,Candidate,NN prediction,NN disagreement,Distance from best,Nearest distance
0,0.03,"[0.708744, 0.167838]",0.640225,0.018623,0.03,0.010164
1,0.05,"[0.707154, 0.187801]",0.638230,0.021073,0.05,0.029790
2,0.10,"[0.704062, 0.237746]",0.617255,0.039456,0.10,0.079742


## Final validation and submission

In [16]:
import numpy as np
final_candidates = {1: np.array([0.002495, 0.694432]), 2: np.array([0.702093, 0.18693]), 3: np.array([0.440641, 0.482965, 0.435994]), 4: np.array([0.38875, 0.360865, 0.352431, 0.416723]), 5: np.array([1.0, 1.0, 1.0, 0.969985]), 6: np.array([0.444874, 0.347937, 0.53347, 0.742972, 0.0]), 7: np.array([0.145227, 0.29657, 0.489754, 0.259986, 0.333711, 0.644995]), 8: np.array([0.0, 0.221175, 0.182914, 0.081049, 0.771543, 0.527149, 0.198086, 0.585768])}
all_passed = True
for i in range(1, 9):
    candidate = final_candidates[i]
    df = data[i]
    X = df.iloc[:, :-1].values
    correct_dims = len(candidate) == X.shape[1]
    within_bounds = np.all((candidate >= 0) & (candidate <= 1))
    distances = np.linalg.norm(X - candidate, axis=1)
    nearest_idx = np.argmin(distances)
    nearest_distance = distances[nearest_idx]
    duplicate = np.any(np.all(np.isclose(X, candidate, atol=1e-06), axis=1))
    passed = correct_dims and within_bounds and (not duplicate)
    if not passed:
        all_passed = False
for i in range(1, 9):
    values = ', '.join((f'{x:.6f}' for x in final_candidates[i]))

validation_rows = []
for i in range(1, 9):
    candidate = final_candidates[i]
    X = data[i].iloc[:, :-1].to_numpy(dtype=float)
    distances = np.linalg.norm(X - candidate, axis=1)
    is_duplicate = np.any(np.all(np.isclose(X, candidate, atol=1e-6), axis=1))
    validation_rows.append({"Function": f"F{i}",
                            "Candidate": np.array2string(candidate, precision=6, separator=", ", max_line_width=200),
                            "Dimensions": len(candidate), "Within bounds": bool(np.all((candidate >= 0) & (candidate <= 1))),
                            "Duplicate": bool(is_duplicate), "Nearest distance": float(np.min(distances)),
                            "Passed": bool(len(candidate)==X.shape[1] and np.all((candidate >= 0) & (candidate <= 1)) and not is_duplicate)})
display(pd.DataFrame(validation_rows).round(6))
print("Final submission points (six decimal places):")
for i in range(1, 9):
    print(f"F{i}: " + "-".join(f"{value:.6f}" for value in final_candidates[i]))

,Function,Candidate,Dimensions,Within bounds,Duplicate,Nearest distance,Passed
0,F1,"[0.002495, 0.694432]",2,True,False,0.301746,True
1,F2,"[0.702093, 0.18693 ]",2,True,False,0.029174,True
2,F3,"[0.440641, 0.482965, 0.435994]",3,True,False,0.032293,True
3,F4,"[0.38875 , 0.360865, 0.352431, 0.416723]",4,True,False,0.007193,True
4,F5,"[1. , 1. , 1. , 0.969985]",4,True,False,0.030015,True
5,F6,"[0.444874, 0.347937, 0.53347 , 0.742972, 0. ...",5,True,False,0.055533,True
6,F7,"[0.145227, 0.29657 , 0.489754, 0.259986, 0.333...",6,True,False,0.047410,True
7,F8,"[0. , 0.221175, 0.182914, 0.081049, 0.771...",8,True,False,0.099346,True


Final submission points (six decimal places):
F1: 0.002495-0.694432
F2: 0.702093-0.186930
F3: 0.440641-0.482965-0.435994
F4: 0.388750-0.360865-0.352431-0.416723
F5: 1.000000-1.000000-1.000000-0.969985
F6: 0.444874-0.347937-0.533470-0.742972-0.000000
F7: 0.145227-0.296570-0.489754-0.259986-0.333711-0.644995
F8: 0.000000-0.221175-0.182914-0.081049-0.771543-0.527149-0.198086-0.585768
